In [1]:
from pprint import pprint
from elasticsearch import Elasticsearch

es = Elasticsearch('http://localhost:9200')
client_info = es.info()
pprint('Connected to Elasticsearch successfully!')
pprint(client_info.body)

'Connected to Elasticsearch successfully!'
{'cluster_name': 'docker-cluster',
 'cluster_uuid': 'WGXTdf8bTw6Y1ejhBBncsA',
 'name': 'c813a54bbd9a',
 'tagline': 'You Know, for Search',
 'version': {'build_date': '2024-08-05T10:05:34.233336849Z',
             'build_flavor': 'default',
             'build_hash': '1a77947f34deddb41af25e6f0ddb8e830159c179',
             'build_snapshot': False,
             'build_type': 'docker',
             'lucene_version': '9.11.1',
             'minimum_index_compatibility_version': '7.0.0',
             'minimum_wire_compatibility_version': '7.17.0',
             'number': '8.15.0'}}


In [5]:
es.indices.delete(index='daisy_1', ignore_unavailable=True)
es.indices.create(index='daisy_1')

es.indices.delete(index='daisy_2', ignore_unavailable=True)
es.indices.create(index='daisy_2')

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'daisy_2'})

In [7]:
import json
from tqdm import tqdm

dummy_data = json.load(open("data/dummy_data.json"))
for document in tqdm(dummy_data, total=len(dummy_data)):
    response = es.index(index = 'daisy_1', body=document)
for document in tqdm(dummy_data, total=len(dummy_data)):
    response = es.index(index='daisy_2', body=document)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 26.58it/s]


### Searching using the Search APIs

In [9]:
response = es.search(
    index='daisy_1',
    body={
        "query": {"match_all": {}}
    }
)

n_hits = response['hits']['total']['value']
print(f"Found {n_hits} documents in daisy_1")

Found 10 documents in daisy_1


In [10]:
response = es.search(
    index='daisy_2',
    body={
        "query": {"match_all": {}}
    }
)

n_hits = response['hits']['total']['value']
print(f"Found {n_hits} documents in daisy_2")

Found 5 documents in daisy_2


#### using wildcard `*` to match multiple indices without listing them individually, such as `"daisy*"`

In [12]:
response = es.search(
    index='daisy*',
    body={
        "query": {"match_all": {}}
    }
)

n_hits = response['hits']['total']['value']
print(f"Found {n_hits} documents in all indexes with name starting with 'daisy'")

Found 15 documents in all indexes with name starting with 'daisy'


#### using `_all` to search all indices

In [14]:
response = es.search(
    index='_all',
    body={
        "query": {"match_all": {}}
    }
)

n_hits = response['hits']['total']['value']
print(f"Found {n_hits} documents in all indexes")

Found 17 documents in all indexes


## Query DSL
- Elasticsearch provides a full Query DSL(Domain Specific Language) based on JSON to define queries.Query DSL consist of two types of clauses.
#### Leaf query clauses:
     Leaf query clauses look for a particular value in a particular field, such as the match, term or range queries.
        These queries can be used by themselves
#### Compound query clauses:
     Compound query clauses wrap other leaf or compound queries and are used to combine multiple queries in a logical fashion 
        such as bool or dis_max query or to alter their behaviour such as the constant_score query.

#### match query

In [20]:
response = es.search(
    index='daisy_1',
    body={
        "query": {
            "match": {
                "text": "Description"
            }
        }
    }
)

n_hits = response['hits']['total']['value']
print(f"Found {n_hits} documents in daisy_1")

Found 10 documents in daisy_1


In [21]:
retrieved_documents = response['hits']['hits']
retrieved_documents

[{'_index': 'daisy_1',
  '_id': 'O_i9RZ4B8a3lsDTdU02r',
  '_score': 0.046520013,
  '_source': {'title': 'Title 1',
   'text': 'Description 1',
   'created_on': '2026-05-01'}},
 {'_index': 'daisy_1',
  '_id': 'PPi9RZ4B8a3lsDTdVE1f',
  '_score': 0.046520013,
  '_source': {'title': 'Title 2',
   'text': 'Description 2',
   'created_on': '2026-05-02'}},
 {'_index': 'daisy_1',
  '_id': 'Pfi9RZ4B8a3lsDTdVE1w',
  '_score': 0.046520013,
  '_source': {'title': 'Title 3',
   'text': 'Description 3',
   'created_on': '2026-05-03'}},
 {'_index': 'daisy_1',
  '_id': 'Pvi9RZ4B8a3lsDTdVE2M',
  '_score': 0.046520013,
  '_source': {'title': 'Title 4',
   'text': 'Description 4',
   'created_on': '2026-05-04'}},
 {'_index': 'daisy_1',
  '_id': 'P_i9RZ4B8a3lsDTdVE2W',
  '_score': 0.046520013,
  '_source': {'title': 'Title 5',
   'text': 'Description 5',
   'created_on': '2026-05-05'}},
 {'_index': 'daisy_1',
  '_id': 'QPi9RZ4B8a3lsDTd2k3y',
  '_score': 0.046520013,
  '_source': {'title': 'Title 1',
   't

#### term query
##### Let's use the Query DSL language to construct a query that will find any document that was created on 2026-05-05

In [23]:
response = es.search(
    index='daisy_1',
    body={
        "query": {
            "term": {
                "created_on": "2026-05-05"
            }
        }
    }
)

n_hits = response['hits']['total']['value']
print(f"Found {n_hits} documents in daisy_1")

Found 2 documents in daisy_1


#### range query
##### Let's find documents that were created before 2026-05-04

In [24]:
response = es.search(
    index='daisy_1',
    body={
        "query": {
            "range": {
                "created_on": {
                    "lte": "2026-05-04"
                }
            }
        }
    }
)

n_hits = response['hits']['total']['value']
print(f"Found {n_hits} documents in daisy_1")

Found 8 documents in daisy_1


#### Compound Clauses
##### to combine leaf clauses together, use the compound clauses
- search for documents that meet the requirement created on `"2026-05-05"` and have the word `Description` in the text field

In [26]:
response = es.search(
    index='daisy_1',
    body={
        "query": {
            "bool": {
                "must": [
                    {
                        "match": {
                            "text": "Description"
                        }
                    },
                    {
                        "range": {
                            "created_on": {
                                "gte": "2026-05-03",
                                "lte": "2026-05-06"
                            }
                        }
                    }
                ]
            }
        }
    }
)

n_hits = response['hits']['total']['value']
print(f"Found {n_hits} documents in daisy_1")

Found 6 documents in daisy_1


In [27]:
retrieved_documents = response['hits']['hits']
retrieved_documents

[{'_index': 'daisy_1',
  '_id': 'Pfi9RZ4B8a3lsDTdVE1w',
  '_score': 1.04652,
  '_source': {'title': 'Title 3',
   'text': 'Description 3',
   'created_on': '2026-05-03'}},
 {'_index': 'daisy_1',
  '_id': 'Pvi9RZ4B8a3lsDTdVE2M',
  '_score': 1.04652,
  '_source': {'title': 'Title 4',
   'text': 'Description 4',
   'created_on': '2026-05-04'}},
 {'_index': 'daisy_1',
  '_id': 'P_i9RZ4B8a3lsDTdVE2W',
  '_score': 1.04652,
  '_source': {'title': 'Title 5',
   'text': 'Description 5',
   'created_on': '2026-05-05'}},
 {'_index': 'daisy_1',
  '_id': 'Qvi9RZ4B8a3lsDTd200V',
  '_score': 1.04652,
  '_source': {'title': 'Title 3',
   'text': 'Description 3',
   'created_on': '2026-05-03'}},
 {'_index': 'daisy_1',
  '_id': 'Q_i9RZ4B8a3lsDTd200i',
  '_score': 1.04652,
  '_source': {'title': 'Title 4',
   'text': 'Description 4',
   'created_on': '2026-05-04'}},
 {'_index': 'daisy_1',
  '_id': 'RPi9RZ4B8a3lsDTd200t',
  '_score': 1.04652,
  '_source': {'title': 'Title 5',
   'text': 'Description 5',
 

In [30]:
import json
dummy_data = json.load(open("data/dummy_data.json"))
for _ in range(7):
    dummy_data += dummy_data

len(dummy_data)

640

#### we have duplicated the dummy data and now use the `bulk api` to index all those documents.

In [31]:
operations = []
for document in dummy_data:
    operations.append({'index': {'_index': 'daisy_1'}})
    operations.append(document)

es.bulk(operations=operations)

ObjectApiResponse({'errors': False, 'took': 16969575, 'items': [{'index': {'_index': 'daisy_1', '_id': 'SvjlRZ4B8a3lsDTdf00r', '_version': 1, 'result': 'created', '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 10, '_primary_term': 1, 'status': 201}}, {'index': {'_index': 'daisy_1', '_id': 'S_jlRZ4B8a3lsDTdf00s', '_version': 1, 'result': 'created', '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 11, '_primary_term': 1, 'status': 201}}, {'index': {'_index': 'daisy_1', '_id': 'TPjlRZ4B8a3lsDTdf00s', '_version': 1, 'result': 'created', '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 12, '_primary_term': 1, 'status': 201}}, {'index': {'_index': 'daisy_1', '_id': 'TfjlRZ4B8a3lsDTdf00s', '_version': 1, 'result': 'created', '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 13, '_primary_term': 1, 'status': 201}}, {'index': {'_index': 'daisy_1', '_id': 'TvjlRZ4B8a3lsDTdf00s', '_version': 1, 'result': 'created', '_shards': {'

## Searching

### Size, From 
- Performing a search that retrieves 10 documents, starting from the 11th document(i.e skipping the first 10 results) to demonstrate the paginationusing the `size` and `from` parameters

In [33]:
response = es.search(
    index="daisy_1",
    body={
        "query": {
            "match_all": {}
        },
        "size": 10,
        "from": 10
    },
)

for hit in response['hits']['hits']:
    print(hit['_source'])

{'title': 'Title 1', 'text': 'Description 1', 'created_on': '2026-05-01'}
{'title': 'Title 2', 'text': 'Description 2', 'created_on': '2026-05-02'}
{'title': 'Title 3', 'text': 'Description 3', 'created_on': '2026-05-03'}
{'title': 'Title 4', 'text': 'Description 4', 'created_on': '2026-05-04'}
{'title': 'Title 5', 'text': 'Description 5', 'created_on': '2026-05-05'}
{'title': 'Title 1', 'text': 'Description 1', 'created_on': '2026-05-01'}
{'title': 'Title 2', 'text': 'Description 2', 'created_on': '2026-05-02'}
{'title': 'Title 3', 'text': 'Description 3', 'created_on': '2026-05-03'}
{'title': 'Title 4', 'text': 'Description 4', 'created_on': '2026-05-04'}
{'title': 'Title 5', 'text': 'Description 5', 'created_on': '2026-05-05'}


### Timeout
- Setting a timeout for the search query, if takes longer than the specified `10s` 10 seconds, it will be aborted.

In [39]:
response = es.search(
    index="daisy_1",
    body={
        "query": {
            "match": {
                "message": "Description"
            }
        },
        "timeout": "10s"
    },
)

response.body

{'took': 4,
 'timed_out': False,
 '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 0, 'relation': 'eq'},
  'max_score': None,
  'hits': []}}

### Aggregation
- Performing an aggregation to calculate the average value of the `age` field across all documents that match the query. The result of the aggregation is stored in the avg_age key.

In [41]:

dummy_data_1 = json.load(open("data/dummy_data_1.json"))
for document in tqdm(dummy_data_1, total=len(dummy_data_1)):
    response = es.index(index = 'daisy_3', body=document)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:01<00:00,  2.86it/s]


In [42]:
response = es.search(
    index='daisy_3',
    body={
        "query": {"match_all": {}}
    }
)

n_hits = response['hits']['total']['value']
print(f"Found {n_hits} documents in daisy_3")

Found 5 documents in daisy_3


In [43]:
response = es.search(
    index="daisy_3",
    body={
        "query": {
            "match_all": {}
        },
        "aggs": {
            "avg_age": {
                "avg": {
                    "field": "age"
                }
            }
        }
    }
)

average_age = response['aggregations']['avg_age']['value']
print(f"Average Age: {average_age}")

Average Age: 31.6


In [45]:
import json
dummy_data_1 = json.load(open("data/dummy_data_1.json"))
for _ in range(4):
    dummy_data_1 += dummy_data_1

len(dummy_data)

640

#### Combining size, from, timeout, and aggs

In [50]:
response = es.search(
    index="daisy_3",
    body={
        "query": {
            "match": {
                "message": "important keyword"
            }
        },
        "aggs": {
            "max_price": {
                "max": {
                    "field": "price"
                }
            }
        },
        "size": 5,
        "from": 10,
        "timeout": "10s"
    },
)

for hit in response['hits']['hits']:
    print(hit['_source'])

max_price = response['aggregations']['max_price']['value']
print(f"Max Price: {max_price}")

Max Price: 200.0
